In [1]:
import os
%pwd

'e:\\Research\\Detecting-Thyroid-Cancer\\research'

In [2]:
os.chdir("../")
%pwd

'e:\\Research\\Detecting-Thyroid-Cancer'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int
    params_classes: int

In [4]:
from ThyroidCancer.constants import *
from ThyroidCancer.utils.common import read_yaml, create_directories, save_json

In [5]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_validation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model=Path(self.config.federated_training.aggregated_model_path),
            # training_data=Path(self.config.data_ingestion.unzip_dir),
            training_data=Path(os.path.join(self.config.data_ingestion.unzip_dir, "Thyroid Data")),
            all_params=self.params,
            params_image_size=self.params.IMAGE_SIZE,
            params_batch_size=self.params.BATCH_SIZE,
            params_classes=self.params.CLASSES
        )
        return eval_config

In [6]:
import tensorflow as tf
from pathlib import Path
from ThyroidCancer.utils.common import save_json

class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):
        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(**datagenerator_kwargs)

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    def load_model(self) -> tf.keras.Model:
        # === REBUILD THE EXACT SAME ARCHITECTURE ===
        base = tf.keras.applications.EfficientNetB0(
            input_shape=self.config.params_image_size,
            weights=None,      # will be overwritten by our weights
            include_top=False
        )

        x = tf.keras.layers.GlobalAveragePooling2D(name="avg_pool")(base.output)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Dropout(0.5, name="top_dropout")(x)
        predictions = tf.keras.layers.Dense(
            units=self.config.params_classes,
            activation="softmax",
            name="predictions"
        )(x)

        model = tf.keras.Model(inputs=base.input, outputs=predictions)

        # === LOAD TRAINED WEIGHTS ===
        print(f"Loading weights from: {self.config.path_of_model}")
        model.load_weights(self.config.path_of_model)

        # === COMPILE (required before evaluate) ===
        model.compile(
            optimizer=tf.keras.optimizers.Adam(),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        print(f"Evaluation model loaded successfully!")
        return model

    def evaluation(self):
        self.model = self.load_model()
        self._valid_generator()
        print("Evaluating on validation set...")
        self.score = self.model.evaluate(self.valid_generator, verbose=1)
        print(f"Validation Loss: {self.score[0]:.4f}")
        print(f"Validation Accuracy: {self.score[1]:.4f}")

    def save_score(self):
        scores = {
            "loss": round(float(self.score[0]), 4),
            "accuracy": round(float(self.score[1]), 4)
        }
        save_json(path=Path("scores.json"), data=scores)
        print("Scores saved to scores.json")

In [7]:
try:
    config = ConfigurationManager()
    val_config = config.get_validation_config()
    evaluation = Evaluation(val_config)
    evaluation.evaluation()
    evaluation.save_score()

except Exception as e:
   raise e

[2025-11-20 22:01:55,154: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-11-20 22:01:55,156: INFO: common: yaml file: params.yaml loaded successfully]
[2025-11-20 22:01:55,160: INFO: common: created directory at: artifacts]
Loading weights from: artifacts\federated_training\aggregated_model.h5
Evaluation model loaded successfully!
Found 934 images belonging to 2 classes.
Evaluating on validation set...
30/30 [==============================] - 26s 682ms/step - loss: 0.7119 - accuracy: 0.5182
Validation Loss: 0.7119
Validation Accuracy: 0.5182
[2025-11-20 22:02:26,132: INFO: common: json file saved at: scores.json]
Scores saved to scores.json
